# Module 11 · Notebook 1 of 3 — Multimodal Fusion: the Toy

**BIOT 6900 · AI-Powered Biotech Workflows**

This is the *worked* notebook. Nothing here is graded — **run it top to bottom and read it.**
Its one job is to make the lecture's §4 claim undeniable on clean, synthetic data,
before Notebook 2 points the same code at real TCGA-BRCA patients.

**The one thing to watch.** We build three data "modalities" for each patient and hide the
outcome's main signal in an *interaction between two of them*. Then we try the three fusion
strategies from the lecture. Keep your eye on **late fusion** — it will lag early and joint,
and the reason is structural, not a bug.

> *Toy by design:* this data is clean, balanced, and complete. That is exactly what real
> clinical data is **not** — Notebook 2 brings the mess (missing modalities, scale mismatch,
> real patients). First, see the machinery work.

**How to run:** `Kernel → Restart & Run All`. Everything is synthetic and in-memory — no
downloads, no data files, no GPU.


In [ ]:
# --- Setup -------------------------------------------------------------------
import warnings; warnings.filterwarnings("ignore")   # keep student output clean
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

RNG = np.random.default_rng(11)   # one seed, fully reproducible
print("numpy", np.__version__, "| pandas", pd.__version__)


## 1 · A synthetic 3-modality cohort (sample-matched)

Each patient has three **feature blocks**, standing in for the real modalities:

| block | stands in for | shape |
|---|---|---|
| `G` | genomics | 6 features |
| `I` | imaging (pre-extracted features, like Notebook 2) | 6 features |
| `C` | clinical | 4 features |

All three share the **same patient axis** — row *i* is the same patient in every block.
That shared axis is what the lecture called *the new shared key: the patient, not the gene*.

**The trick:** the outcome is driven mainly by a **product of one genomics feature and one
imaging feature** (`G0 × I0`) — a signal that lives *between* two modalities and sits inside
*neither*. We add weak within-modality effects and noise on top.


In [ ]:
def make_cohort(n=1400):
    """Three sample-matched feature blocks + a binary outcome whose main signal
    is a CROSS-MODAL interaction (genomics x imaging)."""
    G = RNG.normal(size=(n, 6))   # genomics
    I = RNG.normal(size=(n, 6))   # imaging
    C = RNG.normal(size=(n, 4))   # clinical
    logit = ( 2.4 * (G[:, 0] * I[:, 0])          # <-- the cross-modal interaction (the whole point)
            + 0.7 *  C[:, 0]                       # weak clinical main effect
            + 0.5 *  G[:, 1]                       # weak genomics main effect
            + RNG.normal(scale=0.6, size=n) )      # noise
    y = (logit > 0).astype(int)
    ids = [f"P{idx:04d}" for idx in range(n)]
    return G, I, C, y, ids

G, I, C, y, ids = make_cohort()
print("G (genomics):", G.shape, "| I (imaging):", I.shape, "| C (clinical):", C.shape)
print("outcome balance:", np.bincount(y), " (0 vs 1)")

# a quick peek — the blocks line up on the same patients
preview = pd.DataFrame({"patient": ids[:5],
                        "G0": G[:5,0].round(2), "I0": I[:5,0].round(2),
                        "C0": C[:5,0].round(2), "outcome": y[:5]})
preview


In [ ]:
# one train/test split, reused everywhere so the comparison is fair
idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.30, random_state=11, stratify=y)
print(f"train: {len(tr)} patients | test: {len(te)} patients")

def mlp():
    # a modest nonlinear classifier, identical settings everywhere
    return MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=2000, random_state=0)

scores = {}   # we'll collect every strategy's test AUC here


## 2 · Each modality alone — is any single view enough?

The lecture's thesis: *no single modality sees the whole patient.* Let's measure it. We give
each block its **own** nonlinear model and score it.


In [ ]:
for name, X in [("genomics", G), ("imaging", I), ("clinical", C)]:
    m = mlp().fit(X[tr], y[tr])
    auc = roc_auc_score(y[te], m.predict_proba(X[te])[:, 1])
    scores[f"{name}-only"] = auc
    print(f"{name:>9}-only   AUC = {auc:.3f}")


**What you should see.** Every single modality is weak — AUC not far above 0.5. The clinical
block does best (it carries the only *standalone* effect), but none is usable alone. That's not
because the models are bad; it's because the dominant signal (`G0 × I0`) **isn't present in any
one block.** A model that only ever sees genomics can never see the imaging half of the product.


## 3 · Late fusion — decide separately, then average (the tumor board)

Train one model **per modality**, then combine their *predictions* by averaging. This is the
tumor board from the lecture: independent expert reads, pooled at the end.


In [ ]:
late_probs = []
for X in [G, I, C]:
    m = mlp().fit(X[tr], y[tr])
    late_probs.append(m.predict_proba(X[te])[:, 1])
scores["late"] = roc_auc_score(y[te], np.mean(late_probs, axis=0))
print(f"late fusion      AUC = {scores['late']:.3f}")


**What you should see.** Barely better than the best single modality. Each per-modality model
had to **commit to a prediction before seeing the others** — so no model, and therefore no
average of models, can represent `G0 × I0`. Late fusion's blindness to cross-modal interactions
is *structural*. More training won't fix it.


## 4 · Early fusion — concatenate, then learn once

Glue all three blocks into one wide input and train a **single** nonlinear model on it.


In [ ]:
X_all = np.hstack([G, I, C])
m = mlp().fit(X_all[tr], y[tr])
scores["early"] = roc_auc_score(y[te], m.predict_proba(X_all[te])[:, 1])
print(f"early fusion     AUC = {scores['early']:.3f}")


**What you should see.** A large jump. Now `G0` and `I0` sit in the *same* input vector, so a
nonlinear model can learn their product. Combining **before** the model decides is what lets it
see across modalities.


## 5 · Joint fusion — a learned representation per modality, end to end

Early fusion worked, but it threw every raw feature into one bag. **Joint (intermediate) fusion**
instead learns a small *representation* (an encoder) for each modality, fuses those, and trains
the whole thing end to end.

Here it is as a tiny neural net you can read line by line — **no framework, just NumPy.** Three
encoders (one per modality) → concatenate the learned embeddings → a fusion head → sigmoid.


In [ ]:
def joint_fusion(blocks_tr, y_tr, blocks_eval, k=8, h=16, steps=1500, lr=0.15, seed=0):
    """Minimal end-to-end multimodal net: per-modality encoder -> fuse -> head.
    Everything (encoders + head) is trained jointly by gradient descent."""
    r = np.random.default_rng(seed)
    dims = [b.shape[1] for b in blocks_tr]
    # one linear encoder (d_i -> k) per modality, then one shared fusion head
    Wenc = [r.normal(scale=.3, size=(d, k)) for d in dims]; benc = [np.zeros(k) for _ in dims]
    Wf = r.normal(scale=.3, size=(k*len(dims), h)); bf = np.zeros(h)
    Wo = r.normal(scale=.3, size=(h, 1));           bo = np.zeros(1)
    Y = y_tr.reshape(-1, 1).astype(float)
    relu = lambda z: np.maximum(z, 0)

    for _ in range(steps):
        # ---- forward ----
        E  = [relu(b @ Wenc[i] + benc[i]) for i, b in enumerate(blocks_tr)]  # per-modality embeddings
        F  = np.hstack(E)                                                     # fuse (concatenate embeddings)
        Hh = relu(F @ Wf + bf)                                               # fusion head
        P  = 1 / (1 + np.exp(-(Hh @ Wo + bo)))                               # sigmoid output
        # ---- backward (binary cross-entropy) ----
        dO  = (P - Y) / len(Y)
        dWo = Hh.T @ dO;  dbo = dO.sum(0)
        dHh = dO @ Wo.T;  dHh[Hh <= 0] = 0
        dWf = F.T @ dHh;  dbf = dHh.sum(0)
        dF  = dHh @ Wf.T
        for i in range(len(blocks_tr)):
            dEi = dF[:, i*k:(i+1)*k].copy(); dEi[E[i] <= 0] = 0
            Wenc[i] -= lr * (blocks_tr[i].T @ dEi); benc[i] -= lr * dEi.sum(0)
        Wf -= lr*dWf; bf -= lr*dbf; Wo -= lr*dWo; bo -= lr*dbo

    # ---- predict ----
    E = [relu(b @ Wenc[i] + benc[i]) for i, b in enumerate(blocks_eval)]
    F = np.hstack(E); Hh = relu(F @ Wf + bf)
    return (1 / (1 + np.exp(-(Hh @ Wo + bo)))).ravel()

p_joint = joint_fusion([G[tr], I[tr], C[tr]], y[tr], [G[te], I[te], C[te]])
scores["joint"] = roc_auc_score(y[te], p_joint)
print(f"joint fusion     AUC = {scores['joint']:.3f}")


**What you should see.** On this toy, joint fusion matches early fusion — both see the
interaction. The point isn't that joint wins *here*; it's **how** it got there: it learned a
representation for each modality and combined those, rather than dumping raw features together.
That difference is what pays off in Notebook 2, where the modalities are high-dimensional and
wildly different in shape (a 20k expression vector next to a 1,024-d image embedding).


## 6 · The scoreboard

The whole of lecture §4, in one chart.


In [ ]:
order = ["genomics-only", "imaging-only", "clinical-only", "late", "early", "joint"]
vals  = [scores[k] for k in order]
colors = ["#9A9A9A","#9A9A9A","#9A9A9A","#C58AA0","#D41B2C","#D41B2C"]

fig, ax = plt.subplots(figsize=(9, 4.2))
bars = ax.bar(order, vals, color=colors, edgecolor="#2A2A2A")
ax.axhline(0.5, ls="--", lw=1, color="#888", label="chance (0.5)")
ax.set_ylabel("Test AUC"); ax.set_ylim(0.45, 1.0)
ax.set_title("Where you combine decides what you can see")
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.2f}", ha="center", fontsize=10)
ax.legend(loc="upper left", frameon=False)
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

print(pd.Series(scores, name="AUC").round(3).to_string())


**Read the chart.** Singles and **late fusion** sit near chance for the real signal. **Early**
and **joint** jump, because they can model a feature from genomics *times* a feature from imaging.
Late fusion's gap isn't a tuning problem — it's where the combination happens. *Combine before
the model decides, and you can see across modalities; combine after, and you can't.*


## 7 · Bonus — a missing modality (a teaser for next week)

Real clinics rarely have every modality for every patient. Watch what each strategy does when
**imaging goes missing at test time.**


In [ ]:
# refit the pieces we need, then knock out imaging on the test set
mG = mlp().fit(G[tr], y[tr]); mC = mlp().fit(C[tr], y[tr])

# LATE fusion: just drop imaging's vote and average the survivors — no retraining
late_drop = np.mean([mG.predict_proba(G[te])[:,1], mC.predict_proba(C[te])[:,1]], axis=0)
auc_late_drop = roc_auc_score(y[te], late_drop)

# EARLY fusion: the input is a fixed width, so a missing block must be IMPUTED (mean-filled)
m_early = mlp().fit(X_all[tr], y[tr])
I_missing = np.tile(I[tr].mean(axis=0), (len(te), 1))        # impute imaging with train mean
X_te_imp  = np.hstack([G[te], I_missing, C[te]])
auc_early_imp = roc_auc_score(y[te], m_early.predict_proba(X_te_imp)[:,1])

print(f"late fusion, imaging dropped   AUC = {auc_late_drop:.3f}   (was {scores['late']:.3f})")
print(f"early fusion, imaging imputed   AUC = {auc_early_imp:.3f}   (was {scores['early']:.3f})")


**What you should see.** Late fusion **degrades gracefully** — losing imaging just removes one
vote. Early fusion can't accept a narrower input, so you must **impute** the missing block — and
since imaging was half of the key interaction, imputing it **collapses** early fusion's advantage.

That is the *"handles missing modality"* row of the comparison table, live. It's one of the
central tensions of real multimodal data, and we take it on directly next hour and in Notebook 2.


## The three ideas to carry → Notebook 2

1. **No single modality is enough** when the signal lives *between* modalities.
2. **Where you combine is the decision.** Late = robust but blind to interactions; early/joint = see interactions but brittle to missingness.
3. **Joint fusion learns a representation per modality** — which is what makes it scale to real, heterogeneous data.

**Next:** in Notebook 2 the three blocks stop being synthetic. `G` becomes TCGA-BRCA expression,
`C` becomes real clinical fields (both from UCSC Xena), and `I` becomes the **per-patient CNN
embedding from your Module 10 pathology model**. The same three functions you just read get
pointed at real patients — missingness, scale mismatch, and all.
